In [1]:
# !pip install transformers==4.40.1 bitsandbytes==0.43.1 accelerate==0.29.3 datasets==2.19.0 tiktoken==0.6.0 huggingface_hub==0.22.2 autotrain-advanced==0.7.77 -qqq
!pip install transformers bitsandbytes accelerate datasets tiktoken huggingface_hub autotrain-advanced -qqq
!pip install --upgrade huggingface-hub -qqq

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autotrain-advanced 0.8.36 requires huggingface-hub==0.27.0, but you have huggingface-hub 0.29.1 which is incompatible.


## 예제 6.2. SQL 프롬프트

In [15]:
def make_prompt(ddl, question, query=''):
    prompt = f"""당신은 SQL을 생성하는 SQL 봇입니다. DDL의 테이블을 활용한 Question을 해결할 수 있는 SQL 쿼리를 생성하세요.

### DDL:
{ddl}

### Question:
{question}

### SQL:
{query}"""
    return prompt

## 예제 6.4. 평가를 위한 요청 jsonl 작성 함수

In [25]:
import json
import pandas as pd                                                   
from pathlib import Path

def make_requests_for_gpt_evaluation(df, filename, dir='requests'):
  if not Path(dir).exists():
      Path(dir).mkdir(parents=True)
  prompts = []
  for idx, row in df.iterrows():
      prompts.append("""Based on below DDL and Question, evaluate gen_sql can resolve Question. If gen_sql and gt_sql do equal job, return "yes" else return "no". Output JSON Format: {"resolve_yn": ""}""" + f"""

DDL: {row['context']}
Question: {row['question']}
gt_sql: {row['answer']}
gen_sql: {row['gen_sql']}"""
)

  jobs = [{"model": "gpt-4o-mini", "response_format" : { "type": "json_object" }, "messages": [{"role": "system", "content": prompt}]} for prompt in prompts]
  with open(Path(dir, filename), "w") as f:
      for job in jobs:
          json_string = json.dumps(job)
          f.write(json_string + "\n")

## 예제 6.5. 비동기 요청 명령

In [3]:
import os
# os.environ["OPENAI_API_KEY"] = "자신의 OpenAI API 키 입력"

python api_request_parallel_processor.py \
  --requests_filepath {요청 파일 경로} \
  --save_filepath {생성할 결과 파일 경로} \
  --request_url https://api.openai.com/v1/chat/completions \
  --max_requests_per_minute 300 \
  --max_tokens_per_minute 100000 \
  --token_encoding_name cl100k_base \
  --max_attempts 5 \
  --logging_level 20

SyntaxError: invalid syntax (1231114566.py, line 4)

## 예제 6.6. 결과 jsonl 파일을 csv로 변환하는 함수

In [17]:
def change_jsonl_to_csv(input_file, output_file, prompt_column="prompt", response_column="response"):
    prompts = []
    responses = []
    with open(input_file, 'r') as json_file:
        for data in json_file:
            prompts.append(json.loads(data)[0]['messages'][0]['content'])
            responses.append(json.loads(data)[1]['choices'][0]['message']['content'])

    df = pd.DataFrame({prompt_column: prompts, response_column: responses})
    df.to_csv(output_file, index=False)
    return df

## 예제 6.7. 기초 모델로 생성하기

In [6]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

def make_inference_pipeline(model_id):
  tokenizer = AutoTokenizer.from_pretrained(model_id)
  model = AutoModelForCausalLM.from_pretrained(model_id,
                                               device_map="mps", 
                                               torch_dtype=torch.float16,
                                               # load_in_4bit=True, 
                                               # bnb_4bit_compute_dtype=torch.float16
                                              )
  pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
  return pipe

In [ ]:
model_id = 'beomi/Yi-Ko-6B'
hf_pipe = make_inference_pipeline(model_id)

In [9]:
example = """당신은 SQL을 생성하는 SQL 봇입니다. DDL의 테이블을 활용한 Question을 해결할 수 있는 SQL 쿼리를 생성하세요.

### DDL:
CREATE TABLE players (
  player_id INT PRIMARY KEY AUTO_INCREMENT,
  username VARCHAR(255) UNIQUE NOT NULL,
  email VARCHAR(255) UNIQUE NOT NULL,
  password_hash VARCHAR(255) NOT NULL,
  date_joined DATETIME NOT NULL,
  last_login DATETIME
);

### Question:
사용자 이름에 'admin'이 포함되어 있는 계정의 수를 알려주세요.

### SQL:
"""

In [7]:
%%time
hf_pipe(example, do_sample=False,
    return_full_text=False, max_length=512, truncation=True)
#  SELECT COUNT(*) FROM players WHERE username LIKE '%admin%';

# ### SQL 봇:
# SELECT COUNT(*) FROM players WHERE username LIKE '%admin%';

# ### SQL 봇의 결과:
# SELECT COUNT(*) FROM players WHERE username LIKE '%admin%'; (생략)

CPU times: user 28.8 s, sys: 2.61 s, total: 31.4 s
Wall time: 33.2 s


[{'generated_text': "SELECT COUNT(*) FROM players WHERE username LIKE '%admin%';\n\n### SQL 봇의 결과:\n사용자 이름에 'admin'이 포함되어 있는 계정의 수를 알려주세요.\n\n### SQL 봇의 결과:\nSELECT COUNT(*) FROM players WHERE username LIKE '%admin%';\n\n### SQL 봇의 결과:\n사용자 이름에 'admin'이 포함되어 있는 계정의 수를 알려주세요.\n\n### SQL 봇의 결과:\nSELECT COUNT(*) FROM players WHERE username LIKE '%admin%';\n\n### SQL 봇의 결과:\n사용자 이름에 'admin'이 포함되어 있는 계정의 수를 알려주세요.\n\n### SQL 봇의 결과:\nSELECT COUNT(*) FROM players WHERE username LIKE '%admin%';\n\n### SQL 봇의 결과:\n사용자 이름에 'admin'이 포함되어 있는 계정의 수를 알려주세요.\n\n### SQL 봇의 결과:\nSELECT COUNT(*) FROM players WHERE username LIKE '%admin%';\n\n### SQL 봇의 결과:\n사용자 이름에 'admin'이 포함되어 있는 계정의 수를 알려주세요.\n\n### SQL 봇의 결과:\nSELECT COUNT(*) FROM players WHERE username LIKE '%admin%';\n\n### SQL 봇의 결과:\n사용자 이름에 'admin'이 포함되어 있는 계정의 수를 알려주세요.\n\n### SQL 봇의 결과:\nSELECT COUNT(*) FROM players WHERE username LIKE '%admin%';\n\n### SQL 봇의 결과:\n사용자 이름에 'admin'이 포함되어 있는 계정의 수를 알려주세요.\n\n### SQL 봇의 결과:"}]

## 예제 6.8. 기초 모델 성능 측정

In [8]:
import os
os.environ["TOKENIZERS_PARALLELISM"]="false"
!mkdir results

mkdir: results: File exists


In [13]:
%%time
from datasets import load_dataset
# 데이터셋 불러오기
df = load_dataset("shangrilar/ko_text2sql", "origin")['test']
df = df.to_pandas()

CPU times: user 13.3 ms, sys: 4.7 ms, total: 18 ms
Wall time: 1.99 s


In [18]:
%%time
for idx, row in df.iterrows():
  prompt = make_prompt(row['context'], row['question'])
  df.loc[idx, 'prompt'] = prompt
# sql 생성

CPU times: user 7.21 ms, sys: 306 μs, total: 7.52 ms
Wall time: 7.76 ms


In [19]:
%%time
gen_sqls = hf_pipe(df['prompt'].tolist(), do_sample=False,
                   return_full_text=False, max_length=512, truncation=True)

CPU times: user 4min 50s, sys: 30.3 s, total: 5min 21s
Wall time: 6min 10s


In [20]:
%%time
gen_sqls = [x[0]['generated_text'] for x in gen_sqls]
df['gen_sql'] = gen_sqls

CPU times: user 344 μs, sys: 1.17 ms, total: 1.51 ms
Wall time: 1.51 ms


In [27]:
%%time
# 평가를 위한 requests.jsonl 생성
eval_filepath = "text2sql_evaluation.jsonl"
make_requests_for_gpt_evaluation(df, eval_filepath)

CPU times: user 3.16 ms, sys: 3.03 ms, total: 6.19 ms
Wall time: 5.83 ms


In [28]:
# GPT-4 평가 수행
!python api_request_parallel_processor.py \
--requests_filepath requests/{eval_filepath}  \
--save_filepath results/{eval_filepath} \
--request_url https://api.openai.com/v1/chat/completions \
--max_requests_per_minute 300 \
--max_tokens_per_minute 100000 \
--token_encoding_name cl100k_base \
--max_attempts 5 \
--logging_level 20

INFO:root:Starting request #0
INFO:root:Starting request #1
INFO:root:Starting request #2
INFO:root:Starting request #3
INFO:root:Starting request #4
INFO:root:Starting request #5
INFO:root:Starting request #6
INFO:root:Starting request #7
INFO:root:Starting request #8
INFO:root:Starting request #9
INFO:root:Starting request #10
INFO:root:Starting request #11
INFO:root:Starting request #12
INFO:root:Starting request #13
INFO:root:Starting request #14
INFO:root:Starting request #15
INFO:root:Starting request #16
INFO:root:Starting request #17
INFO:root:Starting request #18
INFO:root:Starting request #19
INFO:root:Starting request #20
INFO:root:Starting request #21
INFO:root:Starting request #22
INFO:root:Starting request #23
INFO:root:Starting request #24
INFO:root:Starting request #25
INFO:root:Starting request #26
INFO:root:Starting request #27
INFO:root:Starting request #28
INFO:root:Starting request #29
INFO:root:Starting request #30
INFO:root:Starting request #31
INFO:root:Starting

In [11]:
base_eval = change_jsonl_to_csv(f"results/{eval_filepath}", "results/yi_ko_6b_eval.csv", "prompt", "resolve_yn")
base_eval['resolve_yn'] = base_eval['resolve_yn'].apply(lambda x: json.loads(x)['resolve_yn'])
num_correct_answers = base_eval.query("resolve_yn == 'yes'").shape[0]
num_correct_answers

21

## 예제 6.9. 학습 데이터 불러오기

In [12]:
from datasets import load_dataset

df_sql = load_dataset("shangrilar/ko_text2sql", "origin")["train"]
df_sql = df_sql.to_pandas()
df_sql = df_sql.dropna().sample(frac=1, random_state=42)
df_sql = df_sql.query("db_id != 1")

In [13]:
for idx, row in df_sql.iterrows():
  df_sql.loc[idx, 'text'] = make_prompt(row['context'], row['question'], row['answer'])

CPU times: user 1.74 s, sys: 11.8 ms, total: 1.75 s
Wall time: 1.77 s


In [14]:
!mkdir data
df_sql.to_csv('data/train.csv', index=False)

## 예제 6.10. 미세 조정 명령어

In [18]:
!autotrain llm --help

usage: autotrain <command> [<args>] llm [-h] [--train] [--deploy]
                                        [--inference] [--backend BACKEND]
                                        [--model MODEL]
                                        [--project-name PROJECT_NAME]
                                        [--data-path DATA_PATH]
                                        [--train-split TRAIN_SPLIT]
                                        [--valid-split VALID_SPLIT]
                                        [--add-eos-token]
                                        [--model-max-length MODEL_MAX_LENGTH]
                                        [--padding PADDING]
                                        [--trainer TRAINER]
                                        [--use-flash-attention-2] [--log LOG]
                                        [--disable-gradient-checkpointing]
                                        [--logging-steps LOGGING_STEPS]
                                        [--eval-strat

In [ ]:
base_model = 'beomi/Yi-Ko-6B'
finetuned_model = 'yi-ko-6b-text2sql'

!autotrain llm \
--train \
--model {base_model} \
--project-name {finetuned_model} \
--data-path data/ \
--text-column text \
--lr 2e-4 \
--batch-size 8 \
--epochs 1 \
--block-size 1024 \
--warmup-ratio 0.1 \
--lora-r 16 \
--lora-alpha 32 \
--lora-dropout 0.05 \
--weight-decay 0.01 \
--gradient-accumulation 8 \
--quantization None \
--mixed-precision bfp16 \
--peft \
# --quantization int4 \
--trainer sft

INFO     | 2025-02-25 14:02:39 | autotrain.cli.run_llm:run:136 - Running LLM
WARNING  | 2025-02-25 14:02:40 | autotrain.trainers.common:__init__:286 - Parameters supplied but not used: config, version, deploy, func, backend, train, inference
Saving the dataset (1/1 shards): 100%|█| 33876/33876 [00:00<00:00, 2024857.03 ex
Saving the dataset (1/1 shards): 100%|█| 33876/33876 [00:00<00:00, 2156775.94 ex
INFO     | 2025-02-25 14:02:40 | autotrain.backends.local:create:20 - Starting local training...
INFO     | 2025-02-25 14:02:40 | autotrain.commands:launch_command:514 - ['accelerate', 'launch', '--num_machines', '1', '--num_processes', '1', '--mixed_precision', 'no', '-m', 'autotrain.trainers.clm', '--training_config', 'yi-ko-6b-text2sql/training_params.json']
INFO     | 2025-02-25 14:02:40 | autotrain.commands:launch_command:515 - {'model': 'beomi/Yi-Ko-6B', 'project_name': 'yi-ko-6b-text2sql', 'data_path': 'yi-ko-6b-text2sql/autotrain-data', 'train_split': 'train', 'valid_split': None, 

## 예제 6.11. LoRA 어댑터 결합 및 허깅페이스 허브 업로드

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel

model_name = base_model
# device_map = {"": 0}
device_map = "mps"

# LoRA와 기초 모델 파라미터 합치기
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map=device_map,
)
model = PeftModel.from_pretrained(base_model, finetuned_model)
model = model.merge_and_unload()

# 토크나이저 설정
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 허깅페이스 허브에 모델 및 토크나이저 저장
model.push_to_hub(finetuned_model, use_temp_dir=False)
tokenizer.push_to_hub(finetuned_model, use_temp_dir=False)

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

/Users/1004506/Work/code/langchain-academy/lc-academy-env/lib/python3.12/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


Upload 3 LFS files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/kimbyl/yi-ko-6b-text2sql/commit/11037446d811d726912ab2c1a05b73751569d356', commit_message='Upload tokenizer', commit_description='', oid='11037446d811d726912ab2c1a05b73751569d356', pr_url=None, repo_url=RepoUrl('https://huggingface.co/kimbyl/yi-ko-6b-text2sql', endpoint='https://huggingface.co', repo_type='model', repo_id='kimbyl/yi-ko-6b-text2sql'), pr_revision=None, pr_num=None)

## 예제 6.12. 미세 조정한 모델로 예시 데이터에 대한 SQL 생성

In [10]:
model_id = "kimbyl/yi-ko-6b-text2sql"
hf_pipe = make_inference_pipeline(model_id)

hf_pipe(example, do_sample=False,
       return_full_text=False, max_length=1024, truncation=True)
# SELECT COUNT(*) FROM players WHERE username LIKE '%admin%';

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use mps


[{'generated_text': "SELECT COUNT(*) FROM players WHERE username LIKE '%admin%';"}]

## 예제 6.13. 미세 조정한 모델 성능 측정

In [35]:
df['gen_sql']

0      SELECT reward_items, SUM(reward_experience) FR...
1      SELECT COUNT(*) FROM players WHERE username LI...
2      SELECT T1.name, T2.reward_experience FROM ques...
3      SELECT name FROM characters WHERE experience >...
4      SELECT c.name, s.skill_name FROM characters AS...
                             ...                        
107    SELECT T1.name, T2.reward_items FROM quests AS...
108    SELECT c.name, c.level, c.experience, i.item_n...
109    SELECT role, COUNT(npc_id) FROM npcs GROUP BY ...
110    SELECT p.player_id, e.item_name FROM character...
111    SELECT I.item_name, I.quantity, C.character_cl...
Name: gen_sql, Length: 112, dtype: object

In [ ]:
# sql 생성 수행
gen_sqls = hf_pipe(df['prompt'].tolist(), do_sample=False,
                   return_full_text=False, max_length=1024, truncation=True)
gen_sqls = [x[0]['generated_text'] for x in gen_sqls]
df['gen_sql'] = gen_sqls

# 평가를 위한 requests.jsonl 생성
ft_eval_filepath = "text2sql_evaluation_finetuned.jsonl"
make_requests_for_gpt_evaluation(df, ft_eval_filepath)

In [36]:
# GPT-4 평가 수행
!python api_request_parallel_processor.py \
  --requests_filepath requests/{ft_eval_filepath} \
  --save_filepath results/{ft_eval_filepath} \
  --request_url https://api.openai.com/v1/chat/completions \
  --max_requests_per_minute 2500 \
  --max_tokens_per_minute 100000 \
  --token_encoding_name cl100k_base \
  --max_attempts 5 \
  --logging_level 20

INFO:root:Starting request #0
INFO:root:Starting request #1
INFO:root:Starting request #2
INFO:root:Starting request #3
INFO:root:Starting request #4
INFO:root:Starting request #5
INFO:root:Starting request #6
INFO:root:Starting request #7
INFO:root:Starting request #8
INFO:root:Starting request #9
INFO:root:Starting request #10
INFO:root:Starting request #11
INFO:root:Starting request #12
INFO:root:Starting request #13
INFO:root:Starting request #14
INFO:root:Starting request #15
INFO:root:Starting request #16
INFO:root:Starting request #17
INFO:root:Starting request #18
INFO:root:Starting request #19
INFO:root:Starting request #20
INFO:root:Starting request #21
INFO:root:Starting request #22
INFO:root:Starting request #23
INFO:root:Starting request #24
INFO:root:Starting request #25
INFO:root:Starting request #26
INFO:root:Starting request #27
INFO:root:Starting request #28
INFO:root:Starting request #29
INFO:root:Starting request #30
INFO:root:Starting request #31
INFO:root:Starting

In [37]:
ft_eval = change_jsonl_to_csv(f"results/{ft_eval_filepath}", "results/yi_ko_6b_eval.csv", "prompt", "resolve_yn")
ft_eval['resolve_yn'] = ft_eval['resolve_yn'].apply(lambda x: json.loads(x)['resolve_yn'])
num_correct_answers = ft_eval.query("resolve_yn == 'yes'").shape[0]
num_correct_answers

60